In [ ]:
import json
import numpy as np
from collections import Counter
from google.colab import drive

drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive'

def load(path):
    with open(path) as f:
        return {r['idx'] if 'idx' in r else r['id']: r for r in json.load(f)}

medqa = {
    'flan':     load(f'{PROJECT_DIR}/flan_t5_results.json'),
    'llama':    load(f'{PROJECT_DIR}/llama_results.json'),
    'qwen':     load(f'{PROJECT_DIR}/qwen_results.json'),
    'gemma':    load(f'{PROJECT_DIR}/gemma3n_results.json'),
    'deepseek': load(f'{PROJECT_DIR}/deepseek_results.json'),
}

medmcqa = {
    'flan':  load(f'{PROJECT_DIR}/medmcqa_flan_t5_results.json'),
    'llama': load(f'{PROJECT_DIR}/medmcqa_llama_results.json'),
    'qwen':  load(f'{PROJECT_DIR}/medmcqa_qwen_results.json'),
    'gemma': load(f'{PROJECT_DIR}/medmcqa_gemma3n_results.json'),
    'gpt4o': load(f'{PROJECT_DIR}/medmcqa_gpt4o_results.json'),
}

def is_valid_letter(x):
    return isinstance(x, str) and x in 'ABCD'

def analyze_dataset(data, strong_names, dataset_label):
    print(f"\n{'='*70}")
    print(f"{dataset_label}")
    print(f"{'='*70}\n")

    flan = data['flan']
    ids = sorted(flan.keys())

    print("Per-model wrong-answer distributions (when wrong, which letter picked):")
    print(f"{'Model':<12} {'A':>8} {'B':>8} {'C':>8} {'D':>8}")
    wrong_dists = {}
    for name in strong_names:
        m = data[name]
        wrong_picks = []
        for i in ids:
            r = m[i]
            if r.get('correct') == 0 and is_valid_letter(r.get('pred')):
                wrong_picks.append(r['pred'])
        c = Counter(wrong_picks)
        total = max(1, sum(c.values()))
        dist = {l: c.get(l, 0)/total for l in 'ABCD'}
        wrong_dists[name] = dist
        print(f"  {name:<10}  " + "  ".join([f"{dist[l]*100:>5.1f}%" for l in 'ABCD']))

    all_strong_wrong_ids = []
    for i in ids:
        if all(data[n][i].get('correct') == 0 for n in strong_names):
            all_strong_wrong_ids.append(i)
    print(f"\nAll-strong-wrong subset: {len(all_strong_wrong_ids)} questions")

    observed_unanim = 0
    valid_questions = 0
    for i in all_strong_wrong_ids:
        preds = [data[n][i].get('pred') for n in strong_names]
        if not all(is_valid_letter(p) for p in preds):
            continue
        valid_questions += 1
        if len(set(preds)) == 1:
            observed_unanim += 1
    observed_rate = observed_unanim / max(1, valid_questions)
    print(f"Observed unanimous-wrong: {observed_unanim}/{valid_questions} = {observed_rate*100:.1f}%")

    n_strong = len(strong_names)
    naive_chance = (1/3)**(n_strong - 1)
    print(f"\nNAIVE chance (uniform among 3 distractors): {naive_chance*100:.1f}%  →  Ratio: {observed_rate/naive_chance:.1f}x")

    empirical_chance_per_q = []
    for i in all_strong_wrong_ids:
        gold = data[strong_names[0]][i].get('gold')
        if not is_valid_letter(gold):
            continue
        distractors = [l for l in 'ABCD' if l != gold]
        renorm = {}
        for n in strong_names:
            d = wrong_dists[n]
            total_dist = sum(d[l] for l in distractors)
            if total_dist == 0:
                renorm[n] = {l: 1/3 for l in distractors}
            else:
                renorm[n] = {l: d[l]/total_dist for l in distractors}
        p_unanim = sum(
            np.prod([renorm[n][L] for n in strong_names])
            for L in distractors
        )
        empirical_chance_per_q.append(p_unanim)

    empirical_chance = float(np.mean(empirical_chance_per_q))
    print(f"EMPIRICAL chance (per-model wrong-answer biases): {empirical_chance*100:.1f}%  →  Ratio: {observed_rate/empirical_chance:.1f}x")
    print(f"\nDelta: empirical chance is {empirical_chance/naive_chance:.2f}x the naive baseline")
    print(f"Observed-vs-empirical ratio (the honest claim): {observed_rate/empirical_chance:.1f}x")

    return observed_rate, naive_chance, empirical_chance

medqa_results = analyze_dataset(medqa, ['llama', 'qwen', 'gemma', 'deepseek'], 'MedQA (4 strong: Llama, Qwen, Gemma, DeepSeek)')
medmcqa_results = analyze_dataset(medmcqa, ['llama', 'qwen', 'gemma', 'gpt4o'], 'MedMCQA (4 strong: Llama, Qwen, Gemma, GPT-4o)')

print(f"\n{'='*70}")
print("SUMMARY: NAIVE vs EMPIRICAL chance baselines")
print(f"{'='*70}\n")
print(f"{'Dataset':<10} {'Observed':<12} {'Naive':<12} {'Empirical':<14} {'Honest ratio'}")
print(f"  MedQA      {medqa_results[0]*100:5.1f}%       {medqa_results[1]*100:5.1f}%        {medqa_results[2]*100:5.1f}%          {medqa_results[0]/medqa_results[2]:.1f}x")
print(f"  MedMCQA    {medmcqa_results[0]*100:5.1f}%       {medmcqa_results[1]*100:5.1f}%        {medmcqa_results[2]*100:5.1f}%          {medmcqa_results[0]/medmcqa_results[2]:.1f}x")

audit_results = {
    'experiment': 'empirical_chance_baseline',
    'medqa': {
        'observed_rate': float(medqa_results[0]),
        'naive_chance': float(medqa_results[1]),
        'empirical_chance': float(medqa_results[2]),
        'honest_ratio': float(medqa_results[0]/medqa_results[2])
    },
    'medmcqa': {
        'observed_rate': float(medmcqa_results[0]),
        'naive_chance': float(medmcqa_results[1]),
        'empirical_chance': float(medmcqa_results[2]),
        'honest_ratio': float(medmcqa_results[0]/medmcqa_results[2])
    },
}
with open(f'{PROJECT_DIR}/audit_empirical_chance.json', 'w') as f:
    json.dump(audit_results, f, indent=2)
print(f"\nSaved audit_empirical_chance.json")

In [ ]:
import numpy as np

flan_c = np.array([medqa['flan'][i].get('correct', 0) for i in sorted(medqa['flan'].keys())])
llama_c = np.array([medqa['llama'][i].get('correct', 0) for i in sorted(medqa['llama'].keys())])
qwen_c = np.array([medqa['qwen'][i].get('correct', 0) for i in sorted(medqa['qwen'].keys())])
gemma_c = np.array([medqa['gemma'][i].get('correct', 0) for i in sorted(medqa['gemma'].keys())])
N = len(flan_c)

print(f"=== MedQA mid-tier-only k-intersection (3 strong: Llama, Qwen, Gemma) ===\n")
print(f"N = {N}")
print(f"Marginal accuracies: Flan {flan_c.mean()*100:.1f}%, Llama {llama_c.mean()*100:.1f}%, Qwen {qwen_c.mean()*100:.1f}%, Gemma {gemma_c.mean()*100:.1f}%\n")
print(f"{'k':<6} {'Observed':<12} {'Null mean':<14} {'Null 95% CI':<18} {'Z':<8}")
print("-" * 60)

B = 10000
rng = np.random.default_rng(seed=42)

for k in range(4):
    wrong_count = (llama_c == 0).astype(int) + (qwen_c == 0).astype(int) + (gemma_c == 0).astype(int)
    observed = int(((flan_c == 1) & (wrong_count == k)).sum())
    null = np.zeros(B, dtype=int)
    for b in range(B):
        f_s = rng.permutation(flan_c)
        l_s = rng.permutation(llama_c)
        q_s = rng.permutation(qwen_c)
        g_s = rng.permutation(gemma_c)
        wc = (l_s == 0).astype(int) + (q_s == 0).astype(int) + (g_s == 0).astype(int)
        null[b] = int(((f_s == 1) & (wc == k)).sum())
    nm, ns = null.mean(), null.std()
    lo, hi = np.percentile(null, [2.5, 97.5])
    z = (observed - nm) / ns if ns > 0 else 0
    print(f"k={k}   {observed:<12} {nm:<14.1f} [{lo:.0f}, {hi:.0f}]{'':<6} {z:+.2f}")

all_mid_wrong = (llama_c == 0) & (qwen_c == 0) & (gemma_c == 0)
n_subset = int(all_mid_wrong.sum())
flan_in = int(((flan_c == 1) & all_mid_wrong).sum())
print(f"\n=== All-3-mid-strong-wrong subset (MedQA) ===")
print(f"  Subset size: {n_subset}")
print(f"  Flan right: {flan_in} ({flan_in/max(1,n_subset)*100:.1f}%)")
print(f"  Flan marginal: {flan_c.mean()*100:.1f}%")

from collections import Counter
ids_sorted = sorted(medqa['flan'].keys())
all_mid_wrong_ids = [ids_sorted[i] for i in range(N) if all_mid_wrong[i]]
unanim = 0
total_classified = 0
for i in all_mid_wrong_ids:
    preds = [medqa['llama'][i].get('pred'), medqa['qwen'][i].get('pred'), medqa['gemma'][i].get('pred')]
    if not all(isinstance(p, str) and p in 'ABCD' for p in preds):
        continue
    total_classified += 1
    if len(set(preds)) == 1:
        unanim += 1

print(f"\n=== Mid-tier unanimous-wrong (3 of 3 picked same wrong) ===")
print(f"  {unanim}/{total_classified} = {unanim/max(1,total_classified)*100:.1f}%")
naive_3model = (1/3)**2
print(f"  Naive chance (3 indep picks among 3 distractors): {naive_3model*100:.1f}%")
print(f"  Ratio to naive chance: {(unanim/max(1,total_classified))/naive_3model:.1f}x")

print(f"\n=== COMPARISON ===")
print(f"  4-model (with DeepSeek):   z(k=4) = +9.61, unanimous = 33.6% (8.6x empirical chance)")
print(f"  3-model (mid-tier only):   z(k=3) = above, unanimous = above")
print(f"\nIf 3-model still shows strong bimodal pattern and high unanimous rate, DeepSeek is NOT load-bearing")

In [ ]:
import json
import numpy as np
from collections import Counter

PROJECT_DIR = '/content/drive/MyDrive'

with open(f'{PROJECT_DIR}/medmcqa_sample.json') as f:
    questions = json.load(f)

idx_to_subject = {q['idx']: q['subject'] for q in questions}

ids_all = sorted(medmcqa['flan'].keys())
non_dental_mask = np.array([idx_to_subject.get(i, '') != 'Dental' for i in ids_all])
N_total = len(ids_all)
N_nondental = int(non_dental_mask.sum())

print(f"Total MedMCQA: {N_total}")
print(f"After excluding Dental: {N_nondental} ({N_nondental/N_total*100:.1f}%)")
print(f"Dropped (Dental): {N_total - N_nondental}\n")

def vec(model_dict):
    return np.array([model_dict[i].get('correct', 0) for i in ids_all])[non_dental_mask]

f_c = vec(medmcqa['flan'])
l_c = vec(medmcqa['llama'])
q_c = vec(medmcqa['qwen'])
g_c = vec(medmcqa['gemma'])
gpt_c = vec(medmcqa['gpt4o'])

print(f"Marginal accuracies on non-Dental MedMCQA:")
print(f"  Flan:   {f_c.mean()*100:.1f}%  (full: 26.8%)")
print(f"  Llama:  {l_c.mean()*100:.1f}%  (full: 54.6%)")
print(f"  Qwen:   {q_c.mean()*100:.1f}%  (full: 56.2%)")
print(f"  Gemma:  {g_c.mean()*100:.1f}%  (full: 49.2%)")
print(f"  GPT-4o: {gpt_c.mean()*100:.1f}%  (full: 77.2%)")

print(f"\n=== Non-Dental MedMCQA: k-intersection (4 strong models) ===\n")
print(f"{'k':<6} {'Observed':<12} {'Null mean':<14} {'Null 95% CI':<18} {'Z':<8}")
print("-" * 60)

strong_vecs = {'llama': l_c, 'qwen': q_c, 'gemma': g_c, 'gpt4o': gpt_c}
strong_names = list(strong_vecs.keys())
B = 10000
rng = np.random.default_rng(seed=42)

for k in range(5):
    wrong_count = sum((strong_vecs[s] == 0).astype(int) for s in strong_names)
    observed = int(((f_c == 1) & (wrong_count == k)).sum())
    null = np.zeros(B, dtype=int)
    for b in range(B):
        f_s = rng.permutation(f_c)
        shuffled = {s: rng.permutation(strong_vecs[s]) for s in strong_names}
        wc = sum((shuffled[s] == 0).astype(int) for s in strong_names)
        null[b] = int(((f_s == 1) & (wc == k)).sum())
    nm, ns = null.mean(), null.std()
    lo, hi = np.percentile(null, [2.5, 97.5])
    z = (observed - nm) / ns if ns > 0 else 0
    print(f"k={k}   {observed:<12} {nm:<14.1f} [{lo:.0f}, {hi:.0f}]{'':<6} {z:+.2f}")

all_strong_wrong = (l_c == 0) & (q_c == 0) & (g_c == 0) & (gpt_c == 0)
n_subset = int(all_strong_wrong.sum())
nondental_indices = [ids_all[i] for i in range(N_total) if non_dental_mask[i]]
all_wrong_nondental_indices = [nondental_indices[i] for i in range(N_nondental) if all_strong_wrong[i]]

unanim = 0
total_classified = 0
for i in all_wrong_nondental_indices:
    preds = [medmcqa[n][i].get('pred') for n in strong_names]
    if not all(isinstance(p, str) and p in 'ABCD' for p in preds):
        continue
    total_classified += 1
    if len(set(preds)) == 1:
        unanim += 1

print(f"\n=== Non-Dental all-4-strong-wrong subset ===")
print(f"  Subset size: {n_subset}")
print(f"  Unanimous-wrong: {unanim}/{total_classified} = {unanim/max(1,total_classified)*100:.1f}%")
print(f"  Naive chance: 3.7%  →  Ratio: {(unanim/max(1,total_classified))/(1/27):.1f}x")

print(f"\n=== COMPARISON ===")
print(f"  Full MedMCQA (n=2816):       z(k=4) = +15.41, unanimous = 23.6% (6.4x naive, 5.7x empirical)")
print(f"  Non-Dental (n={N_nondental}):           see results above")

audit_dental = {
    'experiment': 'non_dental_stratified',
    'n_total': N_total,
    'n_non_dental': N_nondental,
    'unanimous_rate_non_dental': unanim/max(1,total_classified) if total_classified else 0,
    'unanimous_rate_full': 0.236,
    'flan_acc_non_dental': float(f_c.mean()),
    'llama_acc_non_dental': float(l_c.mean()),
    'qwen_acc_non_dental': float(q_c.mean()),
    'gemma_acc_non_dental': float(g_c.mean()),
    'gpt4o_acc_non_dental': float(gpt_c.mean()),
}
with open(f'{PROJECT_DIR}/audit_non_dental.json', 'w') as f:
    json.dump(audit_dental, f, indent=2)
print(f"\nSaved audit_non_dental.json")

In [ ]:
import numpy as np
from collections import Counter

PROJECT_DIR = '/content/drive/MyDrive'

def analyze_5wrong_baseline(data, strong_names, dataset_label, ids_key='idx'):
    print(f"\n{'='*70}")
    print(f"{dataset_label}")
    print(f"{'='*70}\n")

    ids = sorted(data['flan'].keys())
    N = len(ids)

    flan_c = np.array([data['flan'][i].get('correct', 0) for i in ids])
    strong_vecs = {n: np.array([data[n][i].get('correct', 0) for i in ids]) for n in strong_names}
    all_strong_wrong = np.ones(N, dtype=bool)
    for n in strong_names:
        all_strong_wrong &= (strong_vecs[n] == 0)

    all_5_wrong = all_strong_wrong & (flan_c == 0)

    trap_subset = all_strong_wrong & (flan_c == 1)

    full_subset = all_strong_wrong

    def unanim_rate(mask):
        """Among questions in mask, what fraction have all strong models picking the same wrong answer?"""
        subset_indices = [ids[i] for i in range(N) if mask[i]]
        unanim = 0
        total = 0
        for q_id in subset_indices:
            preds = [data[n][q_id].get('pred') for n in strong_names]
            if not all(isinstance(p, str) and p in 'ABCD' for p in preds):
                continue
            total += 1
            if len(set(preds)) == 1:
                unanim += 1
        return unanim, total

    u_full, n_full = unanim_rate(full_subset)
    u_5wrong, n_5wrong = unanim_rate(all_5_wrong)
    u_trap, n_trap = unanim_rate(trap_subset)

    print(f"Subset comparison: unanimous-wrong rate among strong models")
    print(f"{'Subset':<45} {'n':>5} {'Unanimous':>11} {'Rate':>8}")
    print("-" * 75)
    print(f"  Full all-{len(strong_names)}-strong-wrong (Flan any){'':<3} {n_full:>5} {u_full:>11} {u_full/max(1,n_full)*100:>7.1f}%")
    print(f"  All-{len(strong_names)+1}-wrong (Flan ALSO wrong){'':<9} {n_5wrong:>5} {u_5wrong:>11} {u_5wrong/max(1,n_5wrong)*100:>7.1f}%")
    print(f"  All-{len(strong_names)}-strong-wrong + Flan RIGHT (trap){'':<3} {n_trap:>5} {u_trap:>11} {u_trap/max(1,n_trap)*100:>7.1f}%")

    from scipy.stats import fisher_exact
    if n_5wrong > 5 and n_trap > 5:
        table = [[u_5wrong, n_5wrong - u_5wrong],
                 [u_trap, n_trap - u_trap]]
        odds, p = fisher_exact(table)
        print(f"\nFisher exact (all-5-wrong vs trap subset unanimous rates):")
        print(f"  Odds ratio: {odds:.2f}")
        print(f"  p-value:    {p:.4f}")

    print(f"\nInterpretation:")
    r_5w = u_5wrong/max(1,n_5wrong)
    r_trap = u_trap/max(1,n_trap)
    if abs(r_5w - r_trap) < 0.05:
        print(f"  All-5-wrong and trap subsets show SIMILAR unanimous rates")
        print(f"  → Convergence is likely driven by question-level structure (hard questions have dominant distractor)")
        print(f"  → Weakens 'capable-LLM-specific shared bias' interpretation")
    elif r_trap > r_5w:
        print(f"  Trap subset shows HIGHER unanimous rate ({r_trap*100:.1f}% vs {r_5w*100:.1f}%)")
        print(f"  → Convergence on questions Flan got lucky on is STRONGER than on questions Flan also failed")
        print(f"  → Strengthens capable-LLM-specific shared bias interpretation")
    else:
        print(f"  All-5-wrong subset shows HIGHER unanimous rate ({r_5w*100:.1f}% vs {r_trap*100:.1f}%)")
        print(f"  → Questions hard for all 5 models trap strong models more uniformly")
        print(f"  → Question-level structure dominates")

    return {
        'full': {'n': n_full, 'unanim': u_full, 'rate': u_full/max(1,n_full)},
        'all_5_wrong': {'n': n_5wrong, 'unanim': u_5wrong, 'rate': u_5wrong/max(1,n_5wrong)},
        'trap': {'n': n_trap, 'unanim': u_trap, 'rate': u_trap/max(1,n_trap)},
    }

medqa_5w = analyze_5wrong_baseline(medqa, ['llama', 'qwen', 'gemma', 'deepseek'], 'MedQA (4 strong + Flan)')
medmcqa_5w = analyze_5wrong_baseline(medmcqa, ['llama', 'qwen', 'gemma', 'gpt4o'], 'MedMCQA (4 strong + Flan)')

import json
audit_5wrong = {
    'experiment': 'all_5_wrong_baseline',
    'medqa': medqa_5w,
    'medmcqa': medmcqa_5w,
}
with open(f'{PROJECT_DIR}/audit_5wrong_baseline.json', 'w') as f:
    json.dump(audit_5wrong, f, indent=2)
print(f"\nSaved audit_5wrong_baseline.json")

In [ ]:
import numpy as np
from collections import Counter

PROJECT_DIR = '/content/drive/MyDrive'

def permutation_test_lifts(data, model_names, dataset_label):
    print(f"\n{'='*70}")
    print(f"{dataset_label}")
    print(f"{'='*70}\n")

    ids = sorted(data['flan'].keys())
    vecs = {n: np.array([data[n][i].get('correct', 0) for i in ids]) for n in model_names}
    N = len(ids)
    B = 10000
    rng = np.random.default_rng(seed=42)

    def compute_lift(a, b):
        a_wrong_count = (a == 0).sum()
        if a_wrong_count == 0:
            return None
        both_wrong = ((a == 0) & (b == 0)).sum()
        cond = both_wrong / a_wrong_count
        p_b_wrong = (b == 0).mean()
        if p_b_wrong == 0:
            return None
        return cond / p_b_wrong

    print(f"{'Pair':<22} {'Observed':>10} {'Null mean':>12} {'95% null CI':>18} {'Z':>8} {'p':>10}")
    print("-" * 85)

    results = []
    for i, n1 in enumerate(model_names):
        for n2 in model_names[i+1:]:
            a = vecs[n1]
            b = vecs[n2]
            observed = compute_lift(a, b)

            null_lifts = np.zeros(B)
            for s in range(B):
                a_s = rng.permutation(a)
                b_s = rng.permutation(b)
                lift = compute_lift(a_s, b_s)
                null_lifts[s] = lift if lift is not None else 1.0

            nm = null_lifts.mean()
            ns = null_lifts.std()
            lo, hi = np.percentile(null_lifts, [2.5, 97.5])
            z = (observed - nm) / ns if ns > 0 else 0
            p = (null_lifts >= observed).mean()

            kind = 'flan-strong' if 'flan' in (n1, n2) else 'strong-strong'
            print(f"  {n1:>6} ↔ {n2:>6} ({kind})  {observed:>5.2f}x   {nm:>5.2f}x       [{lo:.2f}, {hi:.2f}]      {z:>+6.2f}   {p:>8.4f}")
            results.append({'pair': f'{n1}_{n2}', 'kind': kind, 'observed': observed,
                          'null_mean': float(nm), 'null_ci': [float(lo), float(hi)],
                          'z': float(z), 'p': float(p)})

    print(f"\nSummary:")
    strong_strong = [r for r in results if r['kind'] == 'strong-strong']
    flan_strong = [r for r in results if r['kind'] == 'flan-strong']

    ss_z_range = [r['z'] for r in strong_strong]
    fs_z_range = [r['z'] for r in flan_strong]

    print(f"  Strong-strong pairs: {len(strong_strong)}")
    print(f"    Z range: {min(ss_z_range):+.2f} to {max(ss_z_range):+.2f}")
    print(f"    All p < 0.05: {all(r['p'] < 0.05 for r in strong_strong)}")
    print(f"  Flan-strong pairs:   {len(flan_strong)}")
    print(f"    Z range: {min(fs_z_range):+.2f} to {max(fs_z_range):+.2f}")
    print(f"    All p > 0.05: {all(r['p'] > 0.05 for r in flan_strong)}")

    return results

medqa_lifts = permutation_test_lifts(medqa, ['flan', 'llama', 'qwen', 'gemma', 'deepseek'], 'MedQA — Permutation test on pairwise lifts')
medmcqa_lifts = permutation_test_lifts(medmcqa, ['flan', 'llama', 'qwen', 'gemma', 'gpt4o'], 'MedMCQA — Permutation test on pairwise lifts')

import json
with open(f'{PROJECT_DIR}/audit_pairwise_lift_permutation.json', 'w') as f:
    json.dump({'medqa': medqa_lifts, 'medmcqa': medmcqa_lifts}, f, indent=2)
print(f"\nSaved audit_pairwise_lift_permutation.json")

print(f"\n{'='*70}")
print(f"HEADLINE: Are strong-strong pairs significantly more correlated than Flan-strong?")
print(f"{'='*70}\n")
print(f"Under independence, lift = 1.00 regardless of accuracy. Permutation gives null distribution.")
print(f"If strong-strong all show z > 5 AND flan-strong all show |z| < 2:")
print(f"  → strong models share specific failure mode that Flan does NOT share")
print(f"  → 'Flan as control' interpretation is validated (not just noise)")

In [ ]:
import requests, json, time
from collections import Counter
from datasets import load_dataset
from google.colab import userdata

PROJECT_DIR = '/content/drive/MyDrive'
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

medqa_ds = load_dataset("GBaker/MedQA-USMLE-4-options")
test = medqa_ds['test']
print(f"MedQA test questions: {len(test)}")

print("\n--- Test call ---")
r = requests.post(
    "https://api.openai.com/v1/chat/completions",
    headers={"Authorization": f"Bearer {OPENAI_API_KEY}", "Content-Type": "application/json"},
    json={"model": "gpt-4o",
          "messages": [{"role": "user", "content": "Reply PING"}],
          "max_tokens": 5, "temperature": 0.0},
    timeout=20
)
print(f"Status: {r.status_code}")

if r.status_code != 200:
    print(">>> STOP. Fix OpenAI access. <<<")
else:
    print(">>> Test passed. Running on MedQA... <<<\n")

    def get_answer(question, options, api_key):
        prompt = f"""Answer this USMLE medical question. Reply with only A, B, C, or D.

Question: {question}

A: {options['A']}
B: {options['B']}
C: {options['C']}
D: {options['D']}

Answer:"""
        for attempt in range(3):
            try:
                r = requests.post(
                    "https://api.openai.com/v1/chat/completions",
                    headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
                    json={"model": "gpt-4o",
                          "messages": [{"role": "user", "content": prompt}],
                          "max_tokens": 5, "temperature": 0.0},
                    timeout=30
                )
                if r.status_code == 429:
                    time.sleep(5 * (attempt + 1))
                    continue
                if r.status_code != 200:
                    return None, f"HTTP {r.status_code}"
                text = r.json()['choices'][0]['message']['content'].strip().upper()
                for char in text:
                    if char in 'ABCD':
                        return char, None
                return None, f"No letter: {text!r}"
            except Exception as e:
                if attempt == 2:
                    return None, str(e)
                time.sleep(3)
        return None, "Max retries"

    results = []
    consecutive_errors = 0
    for i, row in enumerate(test):
        pred, err = get_answer(row['question'], row['options'], OPENAI_API_KEY)
        if pred is None:
            consecutive_errors += 1
            if consecutive_errors <= 3:
                print(f"  [{i}] error: {err}")
            if consecutive_errors >= 10:
                print(f"  >>> 10 consecutive errors, aborting <<<")
                break
        else:
            consecutive_errors = 0
        results.append({
            'idx': i,
            'gold': row['answer_idx'],
            'pred': pred,
            'correct': int(pred == row['answer_idx']) if pred else 0,
        })
        if (i + 1) % 200 == 0:
            acc = sum(r['correct'] for r in results) / len(results) * 100
            with open(f'{PROJECT_DIR}/medqa_gpt4o_results_partial.json', 'w') as f:
                json.dump(results, f)
            print(f"  {i+1}/{len(test)} — running acc: {acc:.1f}%")
        time.sleep(0.1)

    total = len(results)
    correct = sum(r['correct'] for r in results)
    print(f"\n=== GPT-4o on MedQA ===")
    print(f"  Accuracy: {correct}/{total} = {correct/total*100:.1f}%")
    print(f"  Compare: DeepSeek-V3 on MedQA was 77.7%")
    pred_dist = Counter(r['pred'] for r in results)
    print(f"  Prediction distribution: {dict(pred_dist)}")
    with open(f'{PROJECT_DIR}/medqa_gpt4o_results.json', 'w') as f:
        json.dump(results, f)
    print(f"  Saved to medqa_gpt4o_results.json")

In [ ]:
import json
import numpy as np
from collections import Counter

PROJECT_DIR = '/content/drive/MyDrive'

def load_results(path):
    with open(path) as f:
        return {r['idx']: r for r in json.load(f)}

flan = load_results(f'{PROJECT_DIR}/flan_t5_results.json')
llama = load_results(f'{PROJECT_DIR}/llama_results.json')
qwen = load_results(f'{PROJECT_DIR}/qwen_results.json')
gemma = load_results(f'{PROJECT_DIR}/gemma3n_results.json')
deepseek = load_results(f'{PROJECT_DIR}/deepseek_results.json')
gpt4o = load_results(f'{PROJECT_DIR}/medqa_gpt4o_results.json')

ids = sorted(flan.keys())
N = len(ids)

def vec(d):
    return np.array([d[i].get('correct', 0) for i in ids])

f_c = vec(flan)
l_c = vec(llama)
q_c = vec(qwen)
g_c = vec(gemma)
d_c = vec(deepseek)
gpt_c = vec(gpt4o)

print(f"=== MedQA marginal accuracies ===")
print(f"  Flan:     {f_c.mean()*100:.1f}%")
print(f"  Llama:    {l_c.mean()*100:.1f}%")
print(f"  Qwen:     {q_c.mean()*100:.1f}%")
print(f"  Gemma:    {g_c.mean()*100:.1f}%")
print(f"  DeepSeek: {d_c.mean()*100:.1f}%")
print(f"  GPT-4o:   {gpt_c.mean()*100:.1f}%")

def run_k_sweep(strong_vecs, strong_names, label):
    print(f"\n=== {label} ===\n")
    print(f"{'k':<6} {'Observed':<12} {'Null mean':<14} {'Null 95% CI':<18} {'Z':<8}")
    print("-" * 60)

    B = 10000
    rng = np.random.default_rng(seed=42)

    for k in range(5):
        wrong_count = sum((strong_vecs[s] == 0).astype(int) for s in strong_names)
        observed = int(((f_c == 1) & (wrong_count == k)).sum())
        null = np.zeros(B, dtype=int)
        for b in range(B):
            f_s = rng.permutation(f_c)
            shuffled = {s: rng.permutation(strong_vecs[s]) for s in strong_names}
            wc = sum((shuffled[s] == 0).astype(int) for s in strong_names)
            null[b] = int(((f_s == 1) & (wc == k)).sum())
        nm, ns = null.mean(), null.std()
        lo, hi = np.percentile(null, [2.5, 97.5])
        z = (observed - nm) / ns if ns > 0 else 0
        print(f"k={k}   {observed:<12} {nm:<14.1f} [{lo:.0f}, {hi:.0f}]{'':<6} {z:+.2f}")

run_k_sweep({'llama': l_c, 'qwen': q_c, 'gemma': g_c, 'deepseek': d_c},
            ['llama', 'qwen', 'gemma', 'deepseek'],
            'MedQA with DeepSeek (original)')

run_k_sweep({'llama': l_c, 'qwen': q_c, 'gemma': g_c, 'gpt4o': gpt_c},
            ['llama', 'qwen', 'gemma', 'gpt4o'],
            'MedQA with GPT-4o (substitution)')

run_k_sweep({'llama': l_c, 'qwen': q_c, 'gemma': g_c, 'deepseek': d_c, 'gpt4o': gpt_c},
            ['llama', 'qwen', 'gemma', 'deepseek', 'gpt4o'],
            'MedQA with BOTH frontier models (5 strong)')

print(f"\n=== Unanimous-wrong (4 strong with GPT-4o) ===")
all_strong_wrong = (l_c == 0) & (q_c == 0) & (g_c == 0) & (gpt_c == 0)
n_subset = int(all_strong_wrong.sum())
all_wrong_ids = [ids[i] for i in range(N) if all_strong_wrong[i]]
unanim, total = 0, 0
for q_id in all_wrong_ids:
    preds = [llama[q_id].get('pred'), qwen[q_id].get('pred'), gemma[q_id].get('pred'), gpt4o[q_id].get('pred')]
    if not all(isinstance(p, str) and p in 'ABCD' for p in preds):
        continue
    total += 1
    if len(set(preds)) == 1:
        unanim += 1
print(f"  All-4-strong-wrong subset: {n_subset}")
print(f"  Unanimous: {unanim}/{total} = {unanim/max(1,total)*100:.1f}%")

print(f"\n=== Unanimous-wrong (5 strong, both frontier models) ===")
all_5_strong_wrong = (l_c == 0) & (q_c == 0) & (g_c == 0) & (d_c == 0) & (gpt_c == 0)
n_5 = int(all_5_strong_wrong.sum())
all_5_wrong_ids = [ids[i] for i in range(N) if all_5_strong_wrong[i]]
unanim5, total5 = 0, 0
for q_id in all_5_wrong_ids:
    preds = [llama[q_id].get('pred'), qwen[q_id].get('pred'), gemma[q_id].get('pred'),
             deepseek[q_id].get('pred'), gpt4o[q_id].get('pred')]
    if not all(isinstance(p, str) and p in 'ABCD' for p in preds):
        continue
    total5 += 1
    if len(set(preds)) == 1:
        unanim5 += 1
print(f"  All-5-strong-wrong subset: {n_5}")
print(f"  Unanimous: {unanim5}/{total5} = {unanim5/max(1,total5)*100:.1f}%")
print(f"  Chance for 5 indep picks among 3 distractors: {(1/3)**4*100:.2f}%")
if unanim5 and total5:
    print(f"  Ratio: {(unanim5/total5)/((1/3)**4):.1f}x")

print(f"\n=== COMPARISON ===")
print(f"  Original (with DeepSeek):    k=4 z=+9.61, unanimous = 33.6% (8.6x empirical)")
print(f"  Substituted (with GPT-4o):   see results above")
print(f"  Extended (both frontier):    see results above")

audit_substitution = {
    'experiment': 'frontier_model_substitution_medqa',
    'medqa_gpt4o_accuracy': float(gpt_c.mean()),
    'all_4_strong_wrong_gpt4o_subset_size': n_subset,
    'unanimous_rate_gpt4o': unanim/max(1,total),
    'all_5_strong_wrong_subset_size': n_5,
    'unanimous_rate_5_strong': unanim5/max(1,total5) if total5 else None,
}
with open(f'{PROJECT_DIR}/audit_frontier_substitution.json', 'w') as f:
    json.dump(audit_substitution, f, indent=2)
print(f"\nSaved audit_frontier_substitution.json")

In [ ]:
import requests, json, time
from collections import Counter
from datasets import load_dataset
from google.colab import userdata

PROJECT_DIR = '/content/drive/MyDrive'
TOGETHER_API_KEY = userdata.get('TOGETHER_API_KEY')

medqa_ds = load_dataset("GBaker/MedQA-USMLE-4-options")
test = medqa_ds['test']

MODELS = {
    'llama_t07':  "meta-llama/Meta-Llama-3-8B-Instruct-Lite",
    'qwen_t07':   "Qwen/Qwen2.5-7B-Instruct-Turbo",
}

def get_answer_majority(question, options, api_key, model, n_samples=5):
    """Sample 5 times at T=0.7, return majority-vote letter."""
    prompt = f"""Answer this USMLE medical question. Reply with only A, B, C, or D.

Question: {question}

A: {options['A']}
B: {options['B']}
C: {options['C']}
D: {options['D']}

Answer:"""
    picks = []
    for _ in range(n_samples):
        try:
            r = requests.post(
                "https://api.together.xyz/v1/chat/completions",
                headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
                json={"model": model,
                      "messages": [{"role": "user", "content": prompt}],
                      "max_tokens": 5, "temperature": 0.7},
                timeout=30
            )
            if r.status_code != 200:
                continue
            text = r.json()['choices'][0]['message']['content'].strip().upper()
            for char in text:
                if char in 'ABCD':
                    picks.append(char)
                    break
        except:
            continue
    if not picks:
        return None, picks
    c = Counter(picks)
    return c.most_common(1)[0][0], picks

for name, model_id in MODELS.items():
    print(f"\n=== {name} ({model_id}) at T=0.7, n=5 samples, majority vote ===")
    print(f"Cost estimate: ~$0.40, runtime ~12 min")

    results = []
    consecutive_errors = 0
    for i, row in enumerate(test):
        pred, samples = get_answer_majority(row['question'], row['options'], TOGETHER_API_KEY, model_id)
        if pred is None:
            consecutive_errors += 1
            if consecutive_errors >= 10:
                print(f"  >>> 10 consecutive errors, aborting <<<")
                break
        else:
            consecutive_errors = 0
        results.append({
            'idx': i,
            'gold': row['answer_idx'],
            'pred': pred,
            'samples': samples,
            'correct': int(pred == row['answer_idx']) if pred else 0,
        })
        if (i + 1) % 200 == 0:
            acc = sum(r['correct'] for r in results) / len(results) * 100
            print(f"  {i+1}/{len(test)} — running acc: {acc:.1f}%")
        time.sleep(0.05)

    total = len(results)
    correct = sum(r['correct'] for r in results)
    print(f"\n  Final accuracy: {correct}/{total} = {correct/total*100:.1f}%")
    with open(f'{PROJECT_DIR}/medqa_{name}_results.json', 'w') as f:
        json.dump(results, f)
    print(f"  Saved to medqa_{name}_results.json")

In [ ]:
import json, numpy as np
from collections import Counter
from google.colab import drive, userdata

drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive'
TOGETHER_API_KEY = userdata.get('TOGETHER_API_KEY')
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

def load_results(path):
    with open(path) as f:
        return {r['idx']: r for r in json.load(f) if 'idx' in r}

flan = load_results(f'{PROJECT_DIR}/flan_t5_results.json')
llama_t0 = load_results(f'{PROJECT_DIR}/llama_results.json')
qwen_t0 = load_results(f'{PROJECT_DIR}/qwen_results.json')
gemma = load_results(f'{PROJECT_DIR}/gemma3n_results.json')
deepseek = load_results(f'{PROJECT_DIR}/deepseek_results.json')
gpt4o_medqa = load_results(f'{PROJECT_DIR}/medqa_gpt4o_results.json')

llama_t07 = load_results(f'{PROJECT_DIR}/medqa_llama_t07_results.json')
qwen_t07 = load_results(f'{PROJECT_DIR}/medqa_qwen_t07_results.json')

print(f"=== Files loaded ===")
print(f"  Flan:        {len(flan)} entries")
print(f"  Llama T=0:   {len(llama_t0)} entries  (acc {sum(r['correct'] for r in llama_t0.values())/len(llama_t0)*100:.1f}%)")
print(f"  Qwen T=0:    {len(qwen_t0)} entries  (acc {sum(r['correct'] for r in qwen_t0.values())/len(qwen_t0)*100:.1f}%)")
print(f"  Gemma:       {len(gemma)} entries")
print(f"  DeepSeek:    {len(deepseek)} entries")
print(f"  GPT-4o:      {len(gpt4o_medqa)} entries")
print(f"  Llama T=0.7: {len(llama_t07)} entries  (acc {sum(r['correct'] for r in llama_t07.values())/len(llama_t07)*100:.1f}%)")
print(f"  Qwen T=0.7:  {len(qwen_t07)} entries  (acc {sum(r['correct'] for r in qwen_t07.values())/len(qwen_t07)*100:.1f}%)")

print(f"\nAPI keys loaded: Together={bool(TOGETHER_API_KEY)}, OpenAI={bool(OPENAI_API_KEY)}")
print(f"\nReady to run Audit Cell 9.")

In [ ]:
def compute_unanim(llama_dict, qwen_dict, gemma_dict, deepseek_dict, label):
    ids = sorted(flan.keys())
    all_strong_wrong_ids = []
    for i in ids:
        if (llama_dict[i].get('correct') == 0
            and qwen_dict[i].get('correct') == 0
            and gemma_dict[i].get('correct') == 0
            and deepseek_dict[i].get('correct') == 0):
            all_strong_wrong_ids.append(i)

    unanim_ids = []
    total_classified = 0
    for q_id in all_strong_wrong_ids:
        preds = [llama_dict[q_id].get('pred'), qwen_dict[q_id].get('pred'),
                 gemma_dict[q_id].get('pred'), deepseek_dict[q_id].get('pred')]
        if not all(isinstance(p, str) and p in 'ABCD' for p in preds):
            continue
        total_classified += 1
        if len(set(preds)) == 1:
            unanim_ids.append(q_id)

    print(f"  {label}: all-4-strong-wrong n={len(all_strong_wrong_ids)}, "
          f"unanimous={len(unanim_ids)}/{total_classified} "
          f"({len(unanim_ids)/max(1,total_classified)*100:.1f}%)")
    return set(unanim_ids), set(all_strong_wrong_ids)

print(f"=== Test-retest: T=0 vs T=0.7 majority-vote ===\n")
print(f"Llama-3-8B accuracy:   T=0 = 52.0%,  T=0.7 majority-vote = 52.6%")
print(f"Qwen-2.5-7B accuracy:  T=0 = 59.5%,  T=0.7 majority-vote = 58.1%\n")

print(f"All-4-strong-wrong subsets and unanimous-wrong counts:")
unanim_t0, subset_t0 = compute_unanim(llama_t0, qwen_t0, gemma, deepseek, 'T=0 (original)')
unanim_t07, subset_t07 = compute_unanim(llama_t07, qwen_t07, gemma, deepseek, 'T=0.7 (Llama+Qwen majority)')

overlap = unanim_t0 & unanim_t07
only_t0 = unanim_t0 - unanim_t07
only_t07 = unanim_t07 - unanim_t0
union = unanim_t0 | unanim_t07
jaccard = len(overlap) / max(1, len(union))

print(f"\n=== Unanimous-wrong set stability ===")
print(f"  T=0 unanimous-wrong:    {len(unanim_t0)}")
print(f"  T=0.7 unanimous-wrong:  {len(unanim_t07)}")
print(f"  Overlap (in both):      {len(overlap)}")
print(f"  Only in T=0:            {len(only_t0)}")
print(f"  Only in T=0.7:          {len(only_t07)}")
print(f"  Jaccard similarity:     {jaccard:.3f} ({jaccard*100:.1f}%)")

subset_overlap = subset_t0 & subset_t07
subset_jaccard = len(subset_overlap) / max(1, len(subset_t0 | subset_t07))
print(f"\n=== All-4-strong-wrong subset stability ===")
print(f"  T=0 subset:             {len(subset_t0)}")
print(f"  T=0.7 subset:           {len(subset_t07)}")
print(f"  Overlap:                {len(subset_overlap)}")
print(f"  Jaccard similarity:     {subset_jaccard:.3f} ({subset_jaccard*100:.1f}%)")

print(f"\n=== Interpretation ===")
if jaccard > 0.7:
    print(f"  HIGH stability — unanimous-wrong finding robust to stochastic decoding")
elif jaccard > 0.4:
    print(f"  MODERATE stability — convergence partially robust, some stochastic component")
else:
    print(f"  LOW stability — concerning T=0 dependence")

audit_t07 = {
    'experiment': 'test_retest_T0_vs_T07_majority_vote',
    'llama_acc_t0': sum(r['correct'] for r in llama_t0.values())/len(llama_t0),
    'llama_acc_t07': sum(r['correct'] for r in llama_t07.values())/len(llama_t07),
    'qwen_acc_t0': sum(r['correct'] for r in qwen_t0.values())/len(qwen_t0),
    'qwen_acc_t07': sum(r['correct'] for r in qwen_t07.values())/len(qwen_t07),
    't0_unanim_count': len(unanim_t0),
    't07_unanim_count': len(unanim_t07),
    'unanim_overlap': len(overlap),
    'unanim_jaccard': float(jaccard),
    'subset_t0_count': len(subset_t0),
    'subset_t07_count': len(subset_t07),
    'subset_overlap': len(subset_overlap),
    'subset_jaccard': float(subset_jaccard),
}

with open(f'{PROJECT_DIR}/audit_test_retest.json', 'w') as f:
    json.dump(audit_t07, f, indent=2)
print(f"\nSaved audit_test_retest.json")

In [ ]:
def compute_unanim(llama_dict, qwen_dict, gemma_dict, deepseek_dict, label):
    ids = sorted(flan.keys())
    all_strong_wrong_ids = []
    for i in ids:
        if (llama_dict[i].get('correct') == 0
            and qwen_dict[i].get('correct') == 0
            and gemma_dict[i].get('correct') == 0
            and deepseek_dict[i].get('correct') == 0):
            all_strong_wrong_ids.append(i)

    unanim_ids = []
    total_classified = 0
    for q_id in all_strong_wrong_ids:
        preds = [llama_dict[q_id].get('pred'), qwen_dict[q_id].get('pred'),
                 gemma_dict[q_id].get('pred'), deepseek_dict[q_id].get('pred')]
        if not all(isinstance(p, str) and p in 'ABCD' for p in preds):
            continue
        total_classified += 1
        if len(set(preds)) == 1:
            unanim_ids.append(q_id)

    print(f"  {label}: all-4-strong-wrong n={len(all_strong_wrong_ids)}, "
          f"unanimous={len(unanim_ids)}/{total_classified} "
          f"({len(unanim_ids)/max(1,total_classified)*100:.1f}%)")
    return set(unanim_ids), set(all_strong_wrong_ids)

print(f"=== Test-retest: T=0 vs T=0.7 majority-vote ===\n")
print(f"Llama-3-8B accuracy:   T=0 = 52.0%,  T=0.7 majority-vote = 52.6%")
print(f"Qwen-2.5-7B accuracy:  T=0 = 59.5%,  T=0.7 majority-vote = 58.1%\n")

print(f"All-4-strong-wrong subsets and unanimous-wrong counts:")
unanim_t0, subset_t0 = compute_unanim(llama_t0, qwen_t0, gemma, deepseek, 'T=0 (original)')
unanim_t07, subset_t07 = compute_unanim(llama_t07, qwen_t07, gemma, deepseek, 'T=0.7 (Llama+Qwen majority)')

overlap = unanim_t0 & unanim_t07
only_t0 = unanim_t0 - unanim_t07
only_t07 = unanim_t07 - unanim_t0
union = unanim_t0 | unanim_t07
jaccard = len(overlap) / max(1, len(union))

print(f"\n=== Unanimous-wrong set stability ===")
print(f"  T=0 unanimous-wrong:    {len(unanim_t0)}")
print(f"  T=0.7 unanimous-wrong:  {len(unanim_t07)}")
print(f"  Overlap (in both):      {len(overlap)}")
print(f"  Only in T=0:            {len(only_t0)}")
print(f"  Only in T=0.7:          {len(only_t07)}")
print(f"  Jaccard similarity:     {jaccard:.3f} ({jaccard*100:.1f}%)")

subset_overlap = subset_t0 & subset_t07
subset_jaccard = len(subset_overlap) / max(1, len(subset_t0 | subset_t07))
print(f"\n=== All-4-strong-wrong subset stability ===")
print(f"  T=0 subset:             {len(subset_t0)}")
print(f"  T=0.7 subset:           {len(subset_t07)}")
print(f"  Overlap:                {len(subset_overlap)}")
print(f"  Jaccard similarity:     {subset_jaccard:.3f} ({subset_jaccard*100:.1f}%)")

print(f"\n=== Interpretation ===")
if jaccard > 0.7:
    print(f"  HIGH stability — unanimous-wrong finding robust to stochastic decoding")
elif jaccard > 0.4:
    print(f"  MODERATE stability — convergence partially robust, some stochastic component")
else:
    print(f"  LOW stability — concerning T=0 dependence")

audit_t07 = {
    'experiment': 'test_retest_T0_vs_T07_majority_vote',
    'llama_acc_t0': sum(r['correct'] for r in llama_t0.values())/len(llama_t0),
    'llama_acc_t07': sum(r['correct'] for r in llama_t07.values())/len(llama_t07),
    'qwen_acc_t0': sum(r['correct'] for r in qwen_t0.values())/len(qwen_t0),
    'qwen_acc_t07': sum(r['correct'] for r in qwen_t07.values())/len(qwen_t07),
    't0_unanim_count': len(unanim_t0),
    't07_unanim_count': len(unanim_t07),
    'unanim_overlap': len(overlap),
    'unanim_jaccard': float(jaccard),
    'subset_t0_count': len(subset_t0),
    'subset_t07_count': len(subset_t07),
    'subset_overlap': len(subset_overlap),
    'subset_jaccard': float(subset_jaccard),
}

with open(f'{PROJECT_DIR}/audit_test_retest.json', 'w') as f:
    json.dump(audit_t07, f, indent=2)
print(f"\nSaved audit_test_retest.json")